# Sprint 11 - Dual-Head Router Test

**Goal:** figure out if routing between two specialists beats the Sprint 10 single-model
(repeat=20). Sprint 10 already passes both gates at accuracy, so the router only earns its
keep if it pushes FIELD accuracy/F1 meaningfully higher (toward the 0.66 field-strong ceiling
and the >= 0.70 stretch target).

**Why this exists:** Sprint 8/9 showed the router's weakness was ROUTING, not the specialists.
Field-strong specialist = 0.66 field / 0.29 lab. Lab-strong = 0.96 lab / 0.41 field. Both are
on Drive. If routing misroutes field images to the lab head, routed field F1 collapses to
~0.30. This notebook measures routing accuracy FIRST (cheap, no training), then decides.

**The bar to beat (Sprint 10):** `best_plantvillage_s10_blr20.pt`
  - Lab accuracy 0.989 / Field accuracy 0.635 (macro F1 field 0.565)

**Baseline specialists on Drive:**
  - Field: `best_plantdoc_stage2_resnet50.pt` (Sprint 7) -> 0.66 field / 0.29 lab
  - Lab:   `best_plantvillage_mixed.pt` (Sprint 5) -> 0.41 field / 0.96 lab
  - Router (dual-head): `best_domain_s9_domain.pt` (Sprint 9, separate backbones)

**Router logic already in code (ml/model.py):**
  - `predict_routed`: domain classifier -> correct head (lab=0 / field=1)
  - `predict_dual`:   confidence-race, higher-confidence head wins (old fallback)

**Plan:**
  1. Setup + hydrate + replay label fixes (label fixes add +0.0287 on top of routing too)
  2. Measure routing accuracy (how often the domain classifier sends PD->field, PV->lab)
  3. Routing gate: >= 95% before trusting end-to-end numbers
  4. Evaluate routed system: field F1 on PD test, lab F1 on PV test (new variants)
  5. Compare to repeat=20 single-model. Only keep router if it actually wins.
  6. If routing accuracy is bad, retrain / improve the domain classifier (Step 8).


In [ ]:
import platform, subprocess, sys
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)


## Step 1 - Mount Drive + clone repo + install deps

Everything lives on Google Drive (checkpoints, results, archives) and the repo is cloned fresh
or pulled. The label-fix replay makes PlantDoc test labels correct every session.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")
LOCAL_RAW_DIR = Path("/content/folium_raw")
LOCAL_DATA_DIR = Path("/content/folium_data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

def run(cmd, cwd, label, stream=False):
    import os, subprocess
    env = dict(os.environ, PYTHONPATH=str(REPO_DIR))
    if stream:
        proc = subprocess.Popen(cmd, cwd=str(cwd), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        lines = []
        for line in proc.stdout:
            print(line, end='')
            lines.append(line)
        proc.wait()
        combined = ''.join(lines)
        if proc.returncode != 0:
            print(f"\n[{label}] failed (returncode {proc.returncode})")
        assert proc.returncode == 0, label
        class _Result:
            def __init__(self, code, out): self.returncode, self.stdout = code, out
        return _Result(proc.returncode, combined)
    proc = subprocess.run(cmd, cwd=str(cwd), env=env, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"[{label}] failed (returncode {proc.returncode})")
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
        assert proc.returncode == 0, label
    return proc


In [ ]:
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn


## Step 2 - Clean old Sprint 9/11 rows from ablation CSV

Only remove `s9_` and this notebook's `s11_` rows so re-runs do not pile up duplicate router
rows in the same variant names. `s10_` and older rows are kept (they are prior evidence).


In [ ]:
import pandas as pd

csv_path = RESULTS_DIR / "ablation_results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    drop = df["variant"].str.startswith(("s9_", "s11_"))
    n = int(drop.sum())
    if n > 0:
        df = df[~drop].reset_index(drop=True)
        df.to_csv(csv_path, index=False)
        print(f"Removed {n} old s9_/s11_ rows")
    else:
        print("No old s9_/s11_ rows found.")
else:
    print("No ablation CSV yet.")


## Step 3 - Hydrate raw from Drive + organize splits + replay label fixes

Same as Sprint 10: hydrate untouched archives, split deterministically (seed 42), then replay
the 7 verified label fixes from `s10_label_fixes.csv` so PlantDoc test labels match the
corrected set.


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--seed", "42",
    "--val-fraction", "0.15",
    "--test-fraction", "0.15",
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("Data ready at", LOCAL_DATA_DIR)


In [ ]:
import pandas as pd
import shutil

fixes_csv = RESULTS_DIR / "s10_label_fixes.csv"
test_dir = LOCAL_DATA_DIR / "plantdoc" / "test"

if fixes_csv.exists():
    fixes = pd.read_csv(fixes_csv)
    moved = done = missing = 0
    for _, row in fixes.iterrows():
        src = test_dir / str(row["src_class"]) / str(row["filename"])
        dst = test_dir / str(row["dst_class"]) / str(row["filename"])
        if dst.exists():
            done += 1
        elif src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            moved += 1
        else:
            missing += 1
            print(f"MISSING: {src}")
    print(f"Label fixes: {moved} applied, {done} already in place, {missing} missing")
else:
    print("No s10_label_fixes.csv found - run the audit once to create it.")


## Step 4 - Locate the router + specialists + repeat=20 baseline

We need the Sprint 9 dual-head checkpoint (contains both heads + the domain classifier) and the
Sprint 10 repeat=20 single model (the bar to beat). The file names are printed so you can
confirm they exist before running anything heavy.


In [ ]:
import torch

names = [
    "best_domain_s9_domain.pt",              # Sprint 9 dual-head router
    "best_plantdoc_stage2_resnet50.pt",      # Sprint 7 field specialist
    "best_plantvillage_mixed.pt",            # Sprint 5 lab specialist
    "best_plantvillage_s10_blr20.pt",        # Sprint 10 repeat=20 single model (BAR TO BEAT)
]
for n in names:
    p = CHECKPOINT_DIR / n
    if p.exists():
        ckpt = torch.load(p, map_location="cpu")
        kw = ckpt.get("model_kwargs", {})
        print(f"OK   {n}  backbone={kw.get('backbone')} classes={len(ckpt.get('class_names', []))} dual?={'separate_backbones' in kw}")
    else:
        print(f"MISS {n}")


## Step 5 - Measure routing accuracy (cheap, no training)

The single most diagnostic number. Load the Sprint 9 dual-head model, run its **domain
classifier** over the PlantDoc test set and a PlantVillage test sample, and report:

- % of PD test images routed to the FIELD head (want ~100%: PD is the field domain)
- % of PV test images routed to the LAB head (want ~100%: PV is the lab domain)

**Routing gate: >= 95% on PD test.** With 230 PD test images, each misroute to the lab head
costs ~0.4% field F1. If PD routing is high but PV is low, the classifier over-specializes to
lab, and you mainly need to retrain it with balanced batches. If PD routing is low, the domain
classifier is basically useless and must be retrained or replaced with confidence-based
selection (predict_dual).


In [ ]:
import sys, torch
sys.path.insert(0, str(REPO_DIR))
from torchvision import transforms
from ml.data_loading import IMAGE_SIZE, IMAGENET_MEAN, IMAGENET_STD

t = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

DOMAIN_CKPT = CHECKPOINT_DIR / "best_domain_s9_domain.pt"
ckpt = torch.load(DOMAIN_CKPT, map_location="cpu")
from ml.model import build_dual_head_model
model = build_dual_head_model(
    num_classes=len(ckpt["class_names"]),
    backbone=ckpt["model_kwargs"].get("backbone", "resnet50"),
    separate_backbones=ckpt["model_kwargs"].get("separate_backbones", False),
)
model.load_state_dict(ckpt["state_dict"])
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Loaded router:", DOMAIN_CKPT.name)
print("Classes:", len(ckpt["class_names"]))
print("Device:", device)


In [ ]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader

def route_proportions(split_dir, limit=None):
    ds = datasets.ImageFolder(split_dir, transform=t)
    if limit is not None and limit < len(ds):
        ds = torch.utils.data.Subset(ds, list(range(limit)))
    loader = DataLoader(ds, batch_size=128, num_workers=2)
    n_lab = n_field = 0
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                feat = model.extract_features(images)
                dom = model.domain_classifier(feat).argmax(dim=1)
            n_lab += (dom == 0).sum().item()
            n_field += (dom == 1).sum().item()
    return n_lab, n_field

pd_dir = LOCAL_DATA_DIR / "plantdoc" / "test"
pv_dir = LOCAL_DATA_DIR / "plantvillage" / "test"

pd_lab, pd_field = route_proportions(pd_dir)
pv_lab, pv_field = route_proportions(pv_dir, limit=2000)

pd_tot, pv_tot = pd_lab + pd_field, pv_lab + pv_field
print("ROUTING ACCURACY  (want PD->field and PV->lab high)")
print(f"  PlantDoc test : {pd_tot} imgs -> lab {pd_lab/pd_tot*100:.1f}% | field {pd_field/pd_tot*100:.1f}%")
print(f"  PlantVillage   : {pv_tot} imgs -> lab {pv_lab/pv_tot*100:.1f}% | field {pv_field/pv_tot*100:.1f}%")

pd_field_frac = pd_field / pd_tot
RATING = "PASS >=95%" if pd_field_frac >= 0.95 else "FAIL <95%"
print(f"  PD -> field: {pd_field_frac*100:.1f}%  [{RATING}]")
print(f"  (est field-F1 penalty from misroutes ~{(1-pd_field_frac)*0.4*100:.1f} pts)")


## Step 6 - Evaluate the routed system end-to-end

If routing accuracy passed (>= 95%), evaluate the Sprint 9 dual-head router on both test sets
with `predict_routed` (and `predict_dual` for comparison). These write NEW `s11_` variant rows
into the ablation CSV. Also evaluate each specialist and the repeat=20 single model for the
comparison table in Step 7.

Variants logged:
  - `s11_router_routed_lab` / `s11_router_routed_field` (domain-classifier route)
  - `s11_router_dual_lab` / `s11_router_dual_field` (confidence-race)
  - `s11_fieldspec_field` (Sprint 7 field specialist on PD, shows the routing ceiling)
  - the repeat=20 baseline is already logged as `s10_blr20_best_*`


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

def eval_row(ckpt_path, dataset, split, variant, extra=(), dual=False):
    args = [
        "--checkpoint", str(ckpt_path),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", split,
        "--variant", variant,
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ] + list(extra)
    if dual:
        args = ["--dual-head"] + args
    cmd = [sys.executable, "-m", "ml.evaluate"] + args
    return run(cmd, cwd=str(REPO_DIR), label=f"eval {variant} failed", stream=True)

ROUTER = CHECKPOINT_DIR / "best_domain_s9_domain.pt"
FIELDSPEC = CHECKPOINT_DIR / "best_plantdoc_stage2_resnet50.pt"

print("6a) ROUTED lab (PlantVillage)")
eval_row(ROUTER, "plantvillage", "test", "s11_router_routed_lab", ["--predict-mode", "routed"], dual=True)
print("6b) ROUTED field (PlantDoc)")
eval_row(ROUTER, "plantdoc", "test", "s11_router_routed_field", ["--map-to-pv", "--predict-mode", "routed"], dual=True)

print("6c) DUAL lab (PlantVillage)")
eval_row(ROUTER, "plantvillage", "test", "s11_router_dual_lab", ["--predict-mode", "dual"], dual=True)
print("6d) DUAL field (PlantDoc)")
eval_row(ROUTER, "plantdoc", "test", "s11_router_dual_field", ["--map-to-pv", "--predict-mode", "dual"], dual=True)

print("6e) Field specialist on PlantDoc (ceiling)")
eval_row(FIELDSPEC, "plantdoc", "test", "s11_fieldspec_field", ["--map-to-pv"])
print("Done evaluating router + specialist.")


## Step 7 - Compare router vs repeat=20 (the bar to beat)

Load the ablation CSV and pull the new `s11_` rows plus the Sprint 10 `s10_blr20_best_*` rows.
The router only wins if routed field accuracy/F1 exceeds repeat=20's field number by more than
noise (~0.03-0.05 on 230 test images) AND lab stays >= repeat=20's lab (0.989).

Decision:
  - routed field > repeat=20 field by >= 0.03 AND routed lab >= 0.95 -> keep router
  - otherwise -> repeat=20 single model stays the shipped checkpoint; router is future work


In [ ]:
import pandas as pd
csv_path = RESULTS_DIR / "ablation_results.csv"
df = pd.read_csv(csv_path)

def row(variant):
    r = df[df["variant"] == variant]
    return r.iloc[0] if len(r) else None

def pretty(variant, name):
    r = row(variant)
    if r is None:
        print(f"  {name:<28} MISSING ({variant})")
        return None
    print(f"  {name:<28} acc={r['accuracy']:.4f}  prec={r['precision']:.4f}  recall={r['recall']:.4f}  f1={r['f1']:.4f}")
    return r

print("=== BAR TO BEAT: Sprint 10 repeat=20 (single model) ===")
field20 = pretty("s10_blr20_best_field", "repeat=20 field")

print("\n=== ROUTER (Sprint 9 dual-head) ===")
pretty("s11_router_routed_lab", "routed lab")
pretty("s11_router_routed_field", "routed field")
pretty("s11_router_dual_lab", "dual lab")
pretty("s11_router_dual_field", "dual field")
pretty("s11_fieldspec_field", "field specialist (PD)")

if field20 is not None:
    target = row("s11_router_routed_field") or row("s11_router_dual_field")
    if target is not None:
        delta = target["accuracy"] - field20["accuracy"]
        print("\n=== DECISION ===")
        print(f"  best router field acc = {target['accuracy']:.4f} vs repeat=20 = {field20['accuracy']:.4f}")
        print(f"  delta = {delta:+.4f}  (need > +0.03 to justify the router)")
        if delta >= 0.03:
            print("  -> ROUTER WINS on field. Keep it if lab stays high; verify on real photos.")
        elif delta >= 0.0:
            print("  -> Router ~equal to repeat=20 on field (within noise). Not worth the complexity.")
        else:
            print("  -> repeat=20 wins. Router does not help; keep single model as shipped checkpoint.")
    else:
        print("  Router field row missing - make sure Step 6 ran without errors.")


## Step 8 - (Conditional) Improve the domain classifier

Only do this if Step 5 routing accuracy was LOW (< 95% PD->field) or Step 7 verdict wanted the
router but it underdelivered. The levers, cheapest first:

  1. **Retrain the domain classifier** with balanced batches: equal PV & PD samples per batch,
     augmentation on the domain task, and features from BOTH backbones concatenated (the current
     classifier only sees the lab backbone). Train via `ml.train.py` with `--dual-head
     --train-head domain`.
  2. **Tune the confidence threshold** instead of a hard 50/50 route (route to field only when
     the domain classifier is confident; else fall back to repeat=20 or the lab head).
  3. **Abandon the router** and keep repeat=20 single-model if these do not beat it after ~2 tries.

This block is commented out by default. Flip it on only after reading the Step 7 verdict.


In [ ]:
# ===== OPTIONAL: retrain the domain classifier (uncomment + set values to run) =====
# DOMAIN_BACKBONE = "resnet50"
# DOMAIN_EPOCHS = "5"
# DOMAIN_LR = "1e-3"
#
# result = run([
#     sys.executable, "-m", "ml.train",
#     "--data-dir", str(LOCAL_DATA_DIR),
#     "--dataset", "plantvillage",
#     "--backbone", DOMAIN_BACKBONE,
#     "--epochs", DOMAIN_EPOCHS,
#     "--lr", DOMAIN_LR,
#     "--batch-size", "64",
#     "--dual-head",
#     "--train-head", "domain",
#     "--separate-backbones",
#     "--tag", "s11_domain",
#     "--checkpoint-dir", str(CHECKPOINT_DIR),
# ], cwd=str(REPO_DIR), label="retrain domain classifier failed")
#
# print("Retrained domain classifier. Re-run Steps 5-7 to see if routing accuracy improved.")


## Summary

This notebook measures the ONE thing that determined whether Sprint 8/9's router could work:
**routing accuracy**. If the domain classifier already routes field photos to the field head
>= 95% of the time, the router is viable and Step 6/7 gives the end-to-end verdict. If it
misroutes, the cheap fix list in Step 8 is the path.

The bar is repeat=20 (already passing both gates). The router only ships if it beats it on
field by a real margin. Otherwise the single model is the shipped checkpoint and the router
stays as documented future work (good for the paper's empirical study angle either way).
